Data Cleaning in Pandas

In [1]:
import pandas as pd

In [3]:
#understand the dataset

df_game_plays = pd.read_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\extract\raw_data\game_plays.csv")
df_game_plays.head(5)
df_game_plays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050529 entries, 0 to 5050528
Data columns (total 18 columns):
 #   Column               Dtype  
---  ------               -----  
 0   play_id              object 
 1   game_id              int64  
 2   team_id_for          float64
 3   team_id_against      float64
 4   event                object 
 5   secondaryType        object 
 6   x                    float64
 7   y                    float64
 8   period               int64  
 9   periodType           object 
 10  periodTime           int64  
 11  periodTimeRemaining  float64
 12  dateTime             object 
 13  goals_away           int64  
 14  goals_home           int64  
 15  description          object 
 16  st_x                 float64
 17  st_y                 float64
dtypes: float64(7), int64(5), object(6)
memory usage: 693.6+ MB


In [5]:
# Create a new dataframe from df_game_plays (copying original to preserve data)
df_clean_game_plays = df_game_plays.copy()

Rename columns 

In [8]:
#Rename columns for consistency 
#Function to add an underscore before uppercase letters and convert to lowercase
import re

def rename_columns(col_name):
    return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', col_name).lower()

# Apply the function to all column names
df_clean_game_plays.columns = [rename_columns(col) for col in df_clean_game_plays.columns]
df_clean_game_plays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5050529 entries, 0 to 5050528
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   play_id                object 
 1   game_id                int64  
 2   team_id_for            float64
 3   team_id_against        float64
 4   event                  object 
 5   secondary_type         object 
 6   x                      float64
 7   y                      float64
 8   period                 int64  
 9   period_type            object 
 10  period_time            int64  
 11  period_time_remaining  float64
 12  date_time              object 
 13  goals_away             int64  
 14  goals_home             int64  
 15  description            object 
 16  st_x                   float64
 17  st_y                   float64
dtypes: float64(7), int64(5), object(6)
memory usage: 693.6+ MB


Check : Null 

In [11]:
# Null - All - check for missing or null values entire dataset
df_clean_game_plays.isnull().sum()

play_id                        0
game_id                        0
team_id_for               932705
team_id_against           932705
event                          0
secondary_type           3868513
x                        1134364
y                        1134333
period                         0
period_type                    0
period_time                    0
period_time_remaining     193019
date_time                      0
goals_away                     0
goals_home                     0
description                    0
st_x                     1134364
st_y                     1134364
dtype: int64

In [12]:
# 'team_id_for' and 'team_id_against' null values, when events are neutral in natural, such as : Game Scheduled, Period Ready, Period Start, Stoppage
# Filter rows where 'team_id_for' or 'team_id_against' are null

# convert dtype to string then replace null value with unknown 

# Filter rows where 'team_id_for' or 'team_id_against' are null or -1
filtered_data = df_clean_game_plays[
    (df_clean_game_plays['team_id_for'].isnull() | (df_clean_game_plays['team_id_for'] == -1)) |
    (df_clean_game_plays['team_id_against'].isnull() | (df_clean_game_plays['team_id_against'] == -1))
][['play_id', 'game_id', 'team_id_for', 'team_id_against', 'event', 'secondary_type', 'description']]


# Display the filtered rows
filtered_data.head(10)

,play_id,game_id,team_id_for,team_id_against,event,secondary_type,description
0,2016020045_1,2016020045,NaN,NaN,Game Scheduled,NaN,Game Scheduled
1,2016020045_2,2016020045,NaN,NaN,Period Ready,NaN,Period Ready
2,2016020045_3,2016020045,NaN,NaN,Period Start,NaN,Period Start
12,2016020045_13,2016020045,NaN,NaN,Stoppage,NaN,Goalie Stopped
21,2016020045_22,2016020045,NaN,NaN,Stoppage,NaN,Goalie Stopped
25,2016020045_26,2016020045,NaN,NaN,Stoppage,NaN,Icing
31,2016020045_32,2016020045,NaN,NaN,Stoppage,NaN,Offside
35,2016020045_36,2016020045,NaN,NaN,Stoppage,NaN,Icing
41,2016020045_42,2016020045,NaN,NaN,Stoppage,NaN,TV timeout
46,2016020045_47,2016020045,NaN,NaN,Stoppage,NaN,Icing


In [15]:
# Handle null values
# Drop rows where 'x' or 'y' columns have NaN values
# df_clean_game_plays = df_clean_game_plays.dropna(subset=['x', 'y'])

Check : Duplicates

In [18]:
# Count unique 'play_id' values
unique_ids = df_clean_game_plays['play_id'].nunique()

# Count total number of rows
total_rows = len(df_clean_game_plays)

# Display the results
print(f"Unique game_ids: {unique_ids}")
print(f"Total rows: {total_rows}")
print(f"There are {total_rows - unique_ids} rows with duplicates.")

# Results: unique_ids 4217063, total_rows 5050529, 833466 rows with duplicates

Unique game_ids: 4217063
Total rows: 5050529
There are 833466 rows with duplicates.


In [20]:
# Additional 2nd Duplicate Check : 'play_id' alone might not be sufficient to determine duplicates 
# use 'game_id', 'team_id_for', 'description'
duplicates_id = df_clean_game_plays[df_clean_game_plays.duplicated(subset=['game_id', 'play_id', 'team_id_for', 'description'], keep=False)]

# Sort rows with duplicated results in ascending order by 'game_id' and 'play_id'
duplicates_id_sorted = duplicates_id.sort_values(by=['game_id', 'play_id', 'team_id_for', 'description'], ascending=True)

# Count total number of rows with duplicates based on 'game_id' and 'play_id'
total_duplicates = df_clean_game_plays.duplicated(subset=['game_id', 'play_id', 'team_id_for', 'description'], keep=False).sum()

# Display the result
print(f"Total number of rows with duplicates based on 'game_id' and 'player_id': {total_duplicates}")

# Display the sorted duplicates for further review
print("Sorted Duplicates based on 'game_id', 'player_id', team_id_for', 'description':")

duplicates_id_sorted.head(10)

Total number of rows with duplicates based on 'game_id' and 'player_id': 1666932
Sorted Duplicates based on 'game_id', 'player_id', team_id_for', 'description':


,play_id,game_id,team_id_for,team_id_against,event,secondary_type,x,y,period,period_type,period_time,period_time_remaining,date_time,goals_away,goals_home,description,st_x,st_y
4162548,2018020001_1,2018020001,NaN,NaN,Game Scheduled,NaN,NaN,NaN,1,REGULAR,0,1200.0,2018-10-03 23:15:05,0,0,Game Scheduled,NaN,NaN
4163825,2018020001_1,2018020001,NaN,NaN,Game Scheduled,NaN,NaN,NaN,1,REGULAR,0,1200.0,2018-10-03 23:15:05,0,0,Game Scheduled,NaN,NaN
4162557,2018020001_10,2018020001,10.0,8.0,Blocked Shot,NaN,56.0,15.0,1,REGULAR,120,1080.0,2018-10-04 00:20:04,0,0,John Tavares blocked shot from Jesperi Kotkaniemi,-56.0,-15.0
4163834,2018020001_10,2018020001,10.0,8.0,Blocked Shot,NaN,56.0,15.0,1,REGULAR,120,1080.0,2018-10-04 00:20:04,0,0,John Tavares blocked shot from Jesperi Kotkaniemi,-56.0,-15.0
4162646,2018020001_100,2018020001,8.0,10.0,Hit,NaN,5.0,-10.0,1,REGULAR,1160,40.0,2018-10-04 00:51:05,1,1,Tomas Tatar hit Auston Matthews,5.0,-10.0
4163925,2018020001_100,2018020001,8.0,10.0,Hit,NaN,5.0,-10.0,1,REGULAR,1160,40.0,2018-10-04 00:51:05,1,1,Tomas Tatar hit Auston Matthews,5.0,-10.0
4162647,2018020001_101,2018020001,10.0,8.0,Hit,NaN,40.0,37.0,1,REGULAR,1168,32.0,2018-10-04 00:51:10,1,1,Ron Hainsey hit Phillip Danault,-40.0,-37.0
4163926,2018020001_101,2018020001,10.0,8.0,Hit,NaN,40.0,37.0,1,REGULAR,1168,32.0,2018-10-04 00:51:10,1,1,Ron Hainsey hit Phillip Danault,-40.0,-37.0
4162648,2018020001_102,2018020001,10.0,8.0,Shot,Snap Shot,-39.0,-28.0,1,REGULAR,1197,3.0,2018-10-04 00:51:39,1,1,Tyler Ennis Snap Shot saved by Carey Price,39.0,28.0
4163927,2018020001_102,2018020001,10.0,8.0,Shot,Snap Shot,-39.0,-28.0,1,REGULAR,1197,3.0,2018-10-04 00:51:39,1,1,Tyler Ennis Snap Shot saved by Carey Price,39.0,28.0


In [21]:
# Remove duplicates based on 'play_id', 'game_id', 'team_id_for', 'description'
df_clean_game_plays = df_clean_game_plays.drop_duplicates(subset=['game_id', 'play_id', 'team_id_for', 'description'], keep='first')

# Display the cleaned dataframe (first 10 rows as an example)
df_clean_game_plays.head(10)

,play_id,game_id,team_id_for,team_id_against,event,secondary_type,x,y,period,period_type,period_time,period_time_remaining,date_time,goals_away,goals_home,description,st_x,st_y
0,2016020045_1,2016020045,NaN,NaN,Game Scheduled,NaN,NaN,NaN,1,REGULAR,0,1200.0,2016-10-18 23:40:58,0,0,Game Scheduled,NaN,NaN
1,2016020045_2,2016020045,NaN,NaN,Period Ready,NaN,NaN,NaN,1,REGULAR,0,1200.0,2016-10-19 01:35:28,0,0,Period Ready,NaN,NaN
2,2016020045_3,2016020045,NaN,NaN,Period Start,NaN,NaN,NaN,1,REGULAR,0,1200.0,2016-10-19 01:40:50,0,0,Period Start,NaN,NaN
3,2016020045_4,2016020045,16.0,4.0,Faceoff,NaN,0.0,0.0,1,REGULAR,0,1200.0,2016-10-19 01:40:50,0,0,Jonathan Toews faceoff won against Claude Giroux,0.0,0.0
4,2016020045_5,2016020045,16.0,4.0,Shot,Wrist Shot,-71.0,9.0,1,REGULAR,54,1146.0,2016-10-19 01:41:44,0,0,Artem Anisimov Wrist Shot saved by Michal Neuv...,71.0,-9.0
5,2016020045_6,2016020045,16.0,4.0,Goal,Wrap-around,-88.0,5.0,1,REGULAR,56,1144.0,2016-10-19 01:41:48,0,1,"Patrick Kane (1) Wrap-around, assists: Artem A...",88.0,-5.0
6,2016020045_7,2016020045,4.0,16.0,Faceoff,NaN,0.0,0.0,1,REGULAR,58,1142.0,2016-10-19 01:42:30,0,1,Pierre-Edouard Bellemare faceoff won against N...,0.0,0.0
7,2016020045_8,2016020045,4.0,16.0,Shot,Wrist Shot,56.0,-7.0,1,REGULAR,69,1131.0,2016-10-19 01:42:41,0,1,Dale Weise Wrist Shot saved by Corey Crawford,56.0,-7.0
8,2016020045_9,2016020045,16.0,4.0,Takeaway,NaN,11.0,21.0,1,REGULAR,78,1122.0,2016-10-19 01:42:49,0,1,Takeaway by Nick Schmaltz,-11.0,-21.0
9,2016020045_10,2016020045,16.0,4.0,Hit,NaN,-68.0,37.0,1,REGULAR,88,1112.0,2016-10-19 01:43:04,0,1,Vinnie Hinostroza hit Brandon Manning,68.0,-37.0


In [22]:
# Final Duplicate Check: Check for duplicates based on 'play_id'

duplicates_id = df_clean_game_plays[df_clean_game_plays.duplicated(subset=['game_id', 'play_id', 'team_id_for', 'description'], keep=False)]

# Sort the duplicated rows by 'game_id', 'player_id', and 'team_id'
duplicates_id_sorted = duplicates_id.sort_values(by=['game_id', 'play_id', 'team_id_for', 'description'], ascending=True)

# Print the sorted duplicated rows
print("Sorted Duplicates based on 'game_id', 'play_id', 'team_id_for' and 'description' :")
print(duplicates_id_sorted[['game_id', 'play_id', 'team_id_for', 'description']])

duplicates_id_sorted.head(50)
# Results : No Duplicates. Resolved!

Sorted Duplicates based on 'game_id', 'play_id', 'team_id_for' and 'description' :
Empty DataFrame
Columns: [game_id, play_id, team_id_for, description]
Index: []


,play_id,game_id,team_id_for,team_id_against,event,secondary_type,x,y,period,period_type,period_time,period_time_remaining,date_time,goals_away,goals_home,description,st_x,st_y


Consistent Column Name - Python PEP Style Guide

In [27]:
df_clean_game_plays.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4217063 entries, 0 to 5050227
Data columns (total 18 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   play_id                object 
 1   game_id                int64  
 2   team_id_for            float64
 3   team_id_against        float64
 4   event                  object 
 5   secondary_type         object 
 6   x                      float64
 7   y                      float64
 8   period                 int64  
 9   period_type            object 
 10  period_time            int64  
 11  period_time_remaining  float64
 12  date_time              object 
 13  goals_away             int64  
 14  goals_home             int64  
 15  description            object 
 16  st_x                   float64
 17  st_y                   float64
dtypes: float64(7), int64(5), object(6)
memory usage: 611.3+ MB


Change Data Type

In [30]:
# Convert non-null float values to integers then strings, and nulls to 'unknown'
df_clean_game_plays['team_id_for'] = df_clean_game_plays['team_id_for'].apply(lambda x: str(int(x)) if pd.notnull(x) else 'unknown')
df_clean_game_plays['team_id_against'] = df_clean_game_plays['team_id_against'].apply(lambda x: str(int(x)) if pd.notnull(x) else 'unknown')

In [31]:
# Change data type syntax - df['column_name'] = df['column_name'].astype('desired_data_type')
# team_id_for, team_id_against, x, y, st_x, st_y, period_time_remaining - int64
import numpy as np

df_clean_game_plays['x'] = df_clean_game_plays['x'].astype('Int64')
df_clean_game_plays['y'] = df_clean_game_plays['y'].astype('Int64')
df_clean_game_plays['st_x'] = df_clean_game_plays['st_x'].astype('Int64')
df_clean_game_plays['st_y'] = df_clean_game_plays['st_y'].astype('Int64')
df_clean_game_plays['period_time_remaining'] = df_clean_game_plays['period_time_remaining'].astype('Int64')
df_clean_game_plays['game_id'] = df_clean_game_plays['game_id'].astype(str)
df_clean_game_plays['team_id_for'] = df_clean_game_plays['team_id_for'].astype(str)
df_clean_game_plays['team_id_against'] = df_clean_game_plays['team_id_against'].astype(str)

In [37]:
# Convert 'game_plays_date_time' from string datatype to datetime datatype

df_clean_game_plays['date_time'] = pd.to_datetime(df_clean_game_plays['date_time'], errors='coerce')

df_clean_game_plays.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4217063 entries, 0 to 5050227
Data columns (total 18 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   play_id                object        
 1   game_id                object        
 2   team_id_for            object        
 3   team_id_against        object        
 4   event                  object        
 5   secondary_type         object        
 6   x                      Int64         
 7   y                      Int64         
 8   period                 int64         
 9   period_type            object        
 10  period_time            int64         
 11  period_time_remaining  Int64         
 12  date_time              datetime64[ns]
 13  goals_away             int64         
 14  goals_home             int64         
 15  description            object        
 16  st_x                   Int64         
 17  st_y                   Int64         
dtypes: Int64(5), datetime64[ns]

In [40]:
#save file locally
#
df_clean_game_plays.to_csv(r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_plays.csv", index=False)